In [1]:
import matplotlib.pyplot as plt
import numpy as np
import subprocess
from ase.optimize.sciopt import *               
from ase.visualize import *
from ase import Atoms
from ase import io
from ase.io import *
from ase.io.cif import read_cif
from ase.io.vasp import write_vasp
from ase.visualize.plot import plot_atoms
from ase.build import add_adsorbate
from ase.io.proteindatabank import read_proteindatabank, write_proteindatabank
from ase.io.lammpsdata import write_lammps_data
from ase.build import bulk, surface
from ase.build.tools import sort

In [3]:
structure = io.read('2335358.cif')
structure = sort(structure)
view(structure)
tmp_molecule=[]
j = 0
num_atoms = len(structure.get_chemical_symbols())
i = 0
del_index = []
N_index = []
flag = False

while i < num_atoms:
    if(structure.get_chemical_symbols()[i] == 'C'):
        del_index.append(i)
        molecule = io.read('FA.pdb')
        molecule.set_cell(structure.cell)
        xmin = molecule.get_center_of_mass()[0]
        xmax = structure.positions[i, 0]
        ymin = molecule.get_center_of_mass()[1]
        ymax = structure.positions[i, 1]
        zmin = molecule.get_center_of_mass()[2]
        zmax = structure.positions[i, 2]
        molecule.positions += (xmax - xmin, ymax - ymin, zmax - zmin)   
        if j==0:
            tmp_molecule = molecule 
        else:
            tmp_molecule += molecule
        j = j+1
    i = i + 1
    
k = 0
while k < num_atoms:
    if(structure.get_chemical_symbols()[k] == 'N'):
        del_index.append(k)
    k = k + 1

del structure[del_index]  


FA_replaced_structure = structure + tmp_molecule
FA_replaced_structure = sort(FA_replaced_structure)

view (FA_replaced_structure)
FA_replaced_structure



/Users/paramvir/miniconda3/lib/python3.8/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(205, setting=1). This may result in wrong setting!
  warnings.warn(


Atoms(symbols='C8H40Br24N16Sn8', pbc=True, cell=[12.0303, 12.0303, 12.0303], atomtypes=..., bfactor=..., occupancy=..., residuenames=..., residuenumbers=..., spacegroup_kinds=...)

In [19]:
import os
from ase.io import read, write
from ase import Atoms
from ase.build import sort

def process_cif_files(root_dir, pdb_file="FA.pdb", output_dir="vasp_output"):
    """
    Process CIF files, replace C with FA molecules, remove N, and save as VASP files.
    
    Parameters:
    root_dir (str): Directory to search for CIF files
    pdb_file (str): Path to FA.pdb file
    output_dir (str): Directory for output VASP files
    """
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    processed_count = 0
    
    # Check if FA.pdb exists
    if not os.path.exists(pdb_file):
        raise FileNotFoundError(f"FA.pdb file not found at: {pdb_file}")
    
    # Read FA molecule template once
    fa_molecule_template = read(pdb_file)
    
    # Walk through directory tree
    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith('.cif'):
                input_path = os.path.join(dirpath, filename)
                output_name = os.path.splitext(filename)[0] + "_FA.vasp"
                output_path = os.path.join(output_dir, output_name)
                
                try:
                    print(f"Processing: {input_path}")
                    
                    # Read and sort initial structure
                    structure = read(input_path)
                    structure = sort(structure)
                    
                    # Initialize variables
                    tmp_molecule = None
                    del_index = []
                    num_atoms = len(structure)
                    i = 0
                    
                    # Process carbon atoms
                    while i < num_atoms:
                        if structure.get_chemical_symbols()[i] == 'C':
                            del_index.append(i)
                            
                            # Create new FA molecule instance
                            molecule = fa_molecule_template.copy()
                            molecule.set_cell(structure.cell)
                            
                            # Position alignment
                            fa_com = molecule.get_center_of_mass()
                            target_pos = structure.positions[i]
                            molecule.positions += (target_pos - fa_com)
                            
                            # Combine molecules
                            if tmp_molecule is None:
                                tmp_molecule = molecule
                            else:
                                tmp_molecule += molecule
                        i += 1
                    
                    # Process nitrogen atoms
                    for k in range(num_atoms):
                        if structure.get_chemical_symbols()[k] == 'N':
                            del_index.append(k)
                    
                    # Delete marked atoms
                    if del_index:
                        del structure[del_index]
                    
                    # Combine structure with FA molecules
                    if tmp_molecule is not None:
                        final_structure = structure + tmp_molecule
                    else:
                        final_structure = structure
                    
                    # Sort final structure
                    final_structure = sort(final_structure)
                    
                    # Write to VASP file
                    write(output_path, final_structure, format='vasp', vasp5=True)
                    print(f"Saved as: {output_path}")
                    processed_count += 1
                    
                except Exception as e:
                    print(f"Error processing {input_path}: {str(e)}")
    
    print(f"\nProcessing complete. Converted {processed_count} CIF files.")

if __name__ == "__main__":
    # Example usage
    root_directory = "."  # Current directory, can be changed
    fa_pdb_file = "FA.pdb"  # Path to your FA.pdb file
    try:
        process_cif_files(root_directory, fa_pdb_file)
    except Exception as e:
        print(f"Error: {str(e)}")

Processing: ./2335363.cif
Saved as: vasp_output/2335363_FA.vasp
Processing: ./2335362.cif
Saved as: vasp_output/2335362_FA.vasp
Processing: ./2335360.cif
Saved as: vasp_output/2335360_FA.vasp
Processing: ./2335361.cif
Saved as: vasp_output/2335361_FA.vasp
Processing: ./2335359.cif
Saved as: vasp_output/2335359_FA.vasp
Processing: ./2335365.cif
Saved as: vasp_output/2335365_FA.vasp
Processing: ./2335370.cif
Saved as: vasp_output/2335370_FA.vasp
Processing: ./2335364.cif
Saved as: vasp_output/2335364_FA.vasp
Processing: ./2335358.cif
Saved as: vasp_output/2335358_FA.vasp
Processing: ./2335366.cif
Saved as: vasp_output/2335366_FA.vasp
Processing: ./2335367.cif
Saved as: vasp_output/2335367_FA.vasp
Processing: ./2335369.cif
Saved as: vasp_output/2335369_FA.vasp
Processing: ./2335368.cif
Saved as: vasp_output/2335368_FA.vasp

Processing complete. Converted 13 CIF files.


In [18]:
from ase.io import read
from ase.visualize import view
atoms = read('vasp_output/2335359_FA.vasp')
view(atoms)

# please make sure Br and I are not overlapping

In [25]:
from ase.io import read
from ase.visualize import view

# Read the VASP file
atoms = read('vasp_output/2335359_FA.vasp')

# Get indices of all atoms except bromine
keep_indices = [i for i, atom in enumerate(atoms) if atom.symbol != 'Br']

# Create new atoms object without bromine
new_atoms = atoms[keep_indices]

# Print some info about the process
original_br_count = sum(1 for atom in atoms if atom.symbol == 'Br')
print(f"Original number of atoms: {len(atoms)}")
print(f"Final number of atoms: {len(new_atoms)}")
print(f"Removed {len(atoms) - len(new_atoms)} bromine atoms")
print(f"Original Br count: {original_br_count}, Final Br count: {sum(1 for atom in new_atoms if atom.symbol == 'Br')}")

# Visualize the result
view(new_atoms)

Original number of atoms: 120
Final number of atoms: 96
Removed 24 bromine atoms
Original Br count: 24, Final Br count: 0


In [49]:
from ase.io import read
from ase.visualize import view
import numpy as np
from random import sample

# Read the VASP file
atoms = read('vasp_output/2335359_FA.vasp')

# Step 1: Remove half of the iodine atoms randomly
i_indices = [i for i, atom in enumerate(atoms) if atom.symbol == 'I']
n_i_remove = len(i_indices) // 2  # Number of iodines to remove (half)
i_to_remove = sample(i_indices, n_i_remove)  # Randomly select half to remove
keep_indices_step1 = [i for i in range(len(atoms)) if i not in i_to_remove]

# Create intermediate atoms object without half of the iodines
atoms_step1 = atoms[keep_indices_step1]

# Step 2: Remove bromines within 0.5 Å of remaining iodines
DISTANCE_THRESHOLD = 0.7 # Distance in Ångstroms
positions = atoms_step1.get_positions()
symbols = atoms_step1.get_chemical_symbols()

# Get indices of remaining iodines and bromines
i_indices_remaining = [i for i, symbol in enumerate(symbols) if symbol == 'I']
br_indices = [i for i, symbol in enumerate(symbols) if symbol == 'Br']

# Identify bromines within 0.5 Å of iodines
br_to_remove = set()
for i_idx in i_indices_remaining:
    pos_i = positions[i_idx]
    for br_idx in br_indices:
        pos_br = positions[br_idx]
        distance = np.linalg.norm(pos_i - pos_br)
        if distance < DISTANCE_THRESHOLD:
            br_to_remove.add(br_idx)

# Keep all atoms except the bromines to remove
keep_indices_step2 = [i for i in range(len(atoms_step1)) if i not in br_to_remove]

# Create final atoms object
new_atoms = atoms_step1[keep_indices_step2]

# Print some info about the process
original_i_count = len(i_indices)
final_i_count = sum(1 for atom in new_atoms if atom.symbol == 'I')
original_br_count = len([i for i in range(len(atoms)) if atoms[i].symbol == 'Br'])
final_br_count = sum(1 for atom in new_atoms if atom.symbol == 'Br')

print(f"Original number of atoms: {len(atoms)}")
print(f"After removing half iodines: {len(atoms_step1)}")
print(f"Final number of atoms: {len(new_atoms)}")
print(f"Original I count: {original_i_count}, Final I count: {final_i_count}")
print(f"Original Br count: {original_br_count}, Final Br count: {final_br_count}")
print(f"Removed {original_i_count - final_i_count} iodine atoms and {original_br_count - final_br_count} bromine atoms")

# Visualize the result
write("2335359_FA_overlapping_removed.vasp", new_atoms, format='vasp', vasp5=True)
view(new_atoms)

Original number of atoms: 120
After removing half iodines: 108
Final number of atoms: 96
Original I count: 24, Final I count: 12
Original Br count: 24, Final Br count: 12
Removed 12 iodine atoms and 12 bromine atoms


In [48]:
from ase.io import read
from ase.visualize import view
import numpy as np
from random import sample

# Read the VASP file
atoms = read('vasp_output/2335360_FA.vasp')

# Ensure periodic boundary conditions are set (assuming they are in the VASP file)
if not atoms.pbc.any():
    raise ValueError("The structure does not have periodic boundary conditions defined.")

# Step 1: Remove half of the bromine atoms randomly
br_indices = [i for i, atom in enumerate(atoms) if atom.symbol == 'Br']
n_br_remove = len(br_indices) // 2  # Number of bromines to remove (half)
br_to_remove = sample(br_indices, n_br_remove)  # Randomly select half to remove
keep_indices_step1 = [i for i in range(len(atoms)) if i not in br_to_remove]

# Create intermediate atoms object without half of the bromines
atoms_step1 = atoms[keep_indices_step1]

# Step 2: Remove iodines overlapping with remaining bromines
DISTANCE_THRESHOLD = 0.8  # Distance in Ångstroms for overlap

# Get indices of remaining bromines and all iodines
symbols = atoms_step1.get_chemical_symbols()
br_indices_remaining = [i for i, symbol in enumerate(symbols) if symbol == 'Br']
i_indices = [i for i, symbol in enumerate(symbols) if symbol == 'I']

# Identify iodines within 0.5 Å of remaining bromines, considering PBC
i_to_remove = set()
for br_idx in br_indices_remaining:
    # Compute distances from this bromine to all iodines with PBC
    distances = atoms_step1.get_distances(br_idx, i_indices, mic=True)
    for i_idx, distance in zip(i_indices, distances):
        if distance < DISTANCE_THRESHOLD:
            i_to_remove.add(i_idx)

# Keep all atoms except the iodines to remove
keep_indices_step2 = [i for i in range(len(atoms_step1)) if i not in i_to_remove]

# Create final atoms object
new_atoms = atoms_step1[keep_indices_step2]

# Print some info about the process
original_br_count = len(br_indices)
final_br_count = sum(1 for atom in new_atoms if atom.symbol == 'Br')
original_i_count = len([i for i in range(len(atoms)) if atoms[i].symbol == 'I'])
final_i_count = sum(1 for atom in new_atoms if atom.symbol == 'I')

print(f"Original number of atoms: {len(atoms)}")
print(f"After removing half bromines: {len(atoms_step1)}")
print(f"Final number of atoms: {len(new_atoms)}")
print(f"Original Br count: {original_br_count}, Final Br count: {final_br_count}")
print(f"Original I count: {original_i_count}, Final I count: {final_i_count}")
print(f"Removed {original_br_count - final_br_count} bromine atoms and {original_i_count - final_i_count} iodine atoms")

# Visualize the result
view(new_atoms)

Original number of atoms: 120
After removing half bromines: 108
Final number of atoms: 96
Original Br count: 24, Final Br count: 12
Original I count: 24, Final I count: 12
Removed 12 bromine atoms and 12 iodine atoms


In [46]:
new_atoms

Atoms(symbols='C8H40Br12I20N16Sn8', pbc=True, cell=[12.018, 12.018, 12.018])